# AegisFin-AI Phase 1 — Rich Prediction Schema + Model Test

This notebook is the **updated integration version**.

It includes the richer schema directly in the notebook and tests the saved `.pkl` model.

It does **not** retrain or tune the model.

The rich schema is split into:
1. customer application information;
2. trusted enrichment information;
3. raw backend overrides.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()

if (PROJECT_ROOT / "app").exists():
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "models" / "aegisfin_phase1_final_model.pkl"

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

In [ ]:
from app.schemas import PredictionRequest, CustomerApplication, EnrichmentData
from app.mapper import request_to_dataframe

print("Customer fields:")
print(list(CustomerApplication.model_fields.keys()))

print("\nEnrichment fields:")
print(list(EnrichmentData.model_fields.keys()))

In [ ]:
# Load the richer sample request.
sample_path = PROJECT_ROOT / "sample_request.json"

with open(sample_path, "r", encoding="utf-8") as f:
    payload = json.load(f)

request = PredictionRequest.model_validate(payload)
raw_df = request_to_dataframe(request)

print("Raw mapped fields:", len(raw_df.columns))
display(raw_df.T)

In [ ]:
# Important: this is the same Phase-1 model service used by FastAPI.
from app.model_service import Phase1ModelService

service = Phase1ModelService(MODEL_PATH)

print(service.info())

probability = service.predict_probability(raw_df)

print(f"\nDefault probability: {probability:.6f}")
print(f"Default probability: {probability * 100:.2f}%")

## Integration architecture

```text
Customer Application
        +
Bureau/Internal Enrichment
        +
Trusted Raw Overrides
        |
        v
     Mapper
        |
        v
Phase 1 Feature Engineering
        |
        v
Saved Preprocessing State
        |
        v
183-model-feature schema
        |
        v
Saved XGBoost
        |
        v
Default Probability
```

Do not retrain the model in this integration notebook.